In [1]:
!pip install -q \
    sentence-transformers \
    faiss-cpu \
    langchain \
    langchain-community \
    langchain-huggingface \
    pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import pandas as pd

posts = pd.read_csv("historical_posts.csv")
posts.head()

,post_id,description,media_type,category,views,likes,saves,shares,reshare
0,1,Ylvie at photo shoot sharing a modeling secret...,reel,modeling,3854,42,2,2,0
1,2,Ylvie in Charleston Swim and Resort Fashion Sh...,reel,modeling,549,8,1,0,0
2,3,Ylvie at Charleston Swim and Resort Fashion Sh...,reel,modeling,2272,17,0,1,0
3,4,Ylvie in princess outfit holding cupcake; anno...,reel,music,285,6,0,0,1
4,5,Ylvie auditioning for Les Miserables twice and...,reel,acting,4083,63,2,2,0


In [3]:
def build_post_text(row: pd.Series) -> str:
    return (
        f"Description: {row['description']}. "
        f"Media type: {row['media_type']}. "
        f"Content category: {row['category']}."
    )

posts["retrieval_text"] = posts.apply(build_post_text, axis=1)

In [5]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

post_embeddings = embedding_model.encode(
    posts["retrieval_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True,
)

post_embeddings = np.asarray(post_embeddings, dtype="float32")

dimension = post_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(post_embeddings)

print(f"Indexed {index.ntotal} historical posts.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Indexed 5 historical posts.


In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [19]:
from pathlib import Path

# MEDIA_FOLDER = Path(
#     "content/drive/MyDrive/Colab Notebooks/IG_Forecaster/"
# )
MEDIA_FOLDER = Path(
    "/content/drive/MyDrive/Colab Notebooks/IG_Forecaster/available_media"
)

MEDIA_FOLDER.mkdir(parents=True, exist_ok=True)

print("Media folder:", MEDIA_FOLDER)

Media folder: /content/drive/MyDrive/Colab Notebooks/IG_Forecaster/available_media


In [29]:
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")


In [21]:
from pydantic import BaseModel, Field
from typing import Literal


class MediaAnalysis(BaseModel):
    file_name: str = Field(
        description="Exact name of the analyzed file."
    )

    media_type: Literal["image", "video"] = Field(
        description="Whether the file is an image or video."
    )

    visual_summary: str = Field(
        description=(
            "A factual description of the visible content, setting, "
            "subject, clothing, objects, and actions."
        )
    )

    themes: list[str] = Field(
        description=(
            "Relevant themes such as music, fashion, rehearsal, "
            "whimsical, professional, or behind the scenes."
        )
    )

    content_categories: list[str] = Field(
        description=(
            "Possible Instagram categories such as music promotion, "
            "performance, fashion, lifestyle, or behind the scenes."
        )
    )

    possible_post_uses: list[str] = Field(
        description=(
            "Specific ways this asset might be used in an Instagram "
            "post, Reel, Story, or carousel."
        )
    )

    quality_notes: str = Field(
        description=(
            "Notes about framing, lighting, clarity, audio, motion, "
            "and whether the asset appears usable."
        )
    )

    spoken_or_sung_content: str | None = Field(
        default=None,
        description=(
            "A brief summary of audible speech or singing in a video. "
            "Use null for images or when no speech or singing is present."
        )
    )

    strongest_moment: str | None = Field(
        default=None,
        description=(
            "For videos, describe the strongest usable moment or hook. "
            "Use null for static images."
        )
    )


In [22]:
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
    ".heic",
}

VIDEO_EXTENSIONS = {
    ".mp4",
    ".mov",
    ".m4v",
    ".avi",
}

SUPPORTED_EXTENSIONS = IMAGE_EXTENSIONS | VIDEO_EXTENSIONS


def find_media_files(folder: Path) -> list[Path]:
    if not folder.exists():
        raise FileNotFoundError(f"Folder does not exist: {folder}")

    files = [
        path
        for path in folder.rglob("*")
        if path.is_file()
        and path.suffix.lower() in SUPPORTED_EXTENSIONS
    ]

    return sorted(files)


media_files = find_media_files(MEDIA_FOLDER)

print(f"Found {len(media_files)} files:")

for path in media_files:
    print("-", path.name)

Found 5 files:
- 37977FB9-109E-4B42-BE52-BF59305E1726.mov
- IMG_6456.MOV
- IMG_6474.MOV
- IMG_6509.MOV
- IMG_6512.MOV


In [23]:
!pip install -q -U google-genai pydantic pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 135.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.1/259.1 kB 25.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.56.3 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.5 which is incompatible.


In [30]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")

if not api_key:
    raise ValueError(
        "Add GEMINI_API_KEY in Colab Secrets and enable notebook access."
    )

client = genai.Client(api_key=api_key)

In [31]:
from typing import Literal
from pydantic import BaseModel, Field


class MediaAnalysis(BaseModel):
    file_name: str
    media_type: Literal["image", "video"]

    visual_summary: str = Field(
        description="Factual summary of the subject, setting, clothing, objects, and actions."
    )

    themes: list[str] = Field(
        description="Themes such as music, fashion, whimsical, rehearsal, or behind the scenes."
    )

    content_categories: list[str] = Field(
        description="Possible Instagram categories such as performance, music promotion, fashion, or lifestyle."
    )

    possible_post_uses: list[str] = Field(
        description="Specific ways this asset could be used in a Reel, carousel, Story, or static post."
    )

    quality_notes: str = Field(
        description="Lighting, framing, clarity, motion, audio, and overall usability."
    )

    spoken_or_sung_content: str | None = Field(
        default=None,
        description="Brief summary of audible speech or singing; null for images or silent videos."
    )

    strongest_moment: str | None = Field(
        default=None,
        description="Strongest usable moment or hook in a video; null for images."
    )

In [42]:
import time
from pathlib import Path
from google.genai import types


MODEL_NAME = "gemini-flash-latest"


def wait_until_ready(uploaded_file, timeout_seconds: int = 600):
    start_time = time.time()
    current_file = uploaded_file

    while True:
        state = getattr(
            current_file.state,
            "name",
            str(current_file.state),
        )

        if state == "ACTIVE":
            return current_file

        if state == "FAILED":
            raise RuntimeError(
                f"Gemini failed to process {current_file.name}"
            )

        if time.time() - start_time > timeout_seconds:
            raise TimeoutError(
                f"Timed out while processing {current_file.name}"
            )

        time.sleep(5)
        current_file = client.files.get(name=current_file.name)


def analyze_media_file(path: Path) -> MediaAnalysis:
    suffix = path.suffix.lower()

    if suffix in IMAGE_EXTENSIONS:
        media_type = "image"
    elif suffix in VIDEO_EXTENSIONS:
        media_type = "video"
    else:
        raise ValueError(f"Unsupported media type: {path}")

    uploaded_file = None

    try:
        print(f"Uploading {path.name}...")

        uploaded_file = client.files.upload(file=str(path))
        uploaded_file = wait_until_ready(uploaded_file)

        prompt = f"""
Analyze this {media_type} as an available asset for an Instagram
content-planning agent.

The account belongs to a young singer, actor, and fashion performer.

Describe only what can reasonably be observed in the media. Do not
invent a song title, campaign, event, date, location, or backstory.

Evaluate:
- visible subject, setting, clothing, objects, and actions
- tone and themes
- possible Instagram content categories
- possible post uses
- lighting, framing, clarity, audio, and usability
- audible speech or singing, when present
- strongest opening or usable moment for video

The exact file name is {path.name}.
The media type is {media_type}.
"""

        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=[uploaded_file, prompt],
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=MediaAnalysis,
                temperature=0.2,
            ),
        )

        analysis = response.parsed

        if analysis is None:
            raise ValueError(
                f"Gemini returned no structured analysis for {path.name}"
            )

        analysis.file_name = path.name
        analysis.media_type = media_type

        return analysis

    finally:
        if uploaded_file is not None:
            try:
                client.files.delete(name=uploaded_file.name)
            except Exception as exc:
                print(
                    f"Temporary upload cleanup failed for "
                    f"{path.name}: {exc}"
                )

In [43]:
test_analysis = analyze_media_file(media_files[0])

print(test_analysis.model_dump_json(indent=2))

Uploading 37977FB9-109E-4B42-BE52-BF59305E1726.mov...
{
  "file_name": "37977FB9-109E-4B42-BE52-BF59305E1726.mov",
  "media_type": "video",
  "visual_summary": "Filmed in vertical orientation but rotated 90 degrees counterclockwise, the video shows a colorful street art mural featuring 'Buff Monster' cartoon ice cream characters. A young woman in a white skirt, white top, and pink sneakers walks into frame and steps along the wall past the vibrant artwork.",
  "themes": [
    "street art",
    "fashion",
    "lifestyle",
    "urban exploration"
  ],
  "content_categories": [
    "fashion",
    "lifestyle",
    "travel"
  ],
  "possible_post_uses": [
    "B-roll clip for an Instagram Reel showcasing street style or city travel",
    "Background visual for an audio snippet or music teaser",
    "Instagram Story post featuring colorful urban backdrops"
  ],
  "quality_notes": "The video is rotated 90 degrees sideways and requires rotation editing prior to use. Features bright daytime ligh

In [5]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Colab Notebooks/IG_Forecaster"
)

gitignore_path = PROJECT_ROOT / ".gitignore"

gitignore_path.write_text(
    """*.gsheet
*.gdoc
*.gslides
.ipynb_checkpoints/
__pycache__/
*.pyc
""",
    encoding="utf-8",
)

print(gitignore_path.read_text())

*.gsheet
*.gdoc
*.gslides
.ipynb_checkpoints/
__pycache__/
*.pyc



In [6]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/IG_Forecaster")

wrong_file = PROJECT_ROOT / "..gitignore"

if wrong_file.exists():
    wrong_file.unlink()
    print("Deleted:", wrong_file)
else:
    print("No incorrect ..gitignore found.")

Deleted: /content/drive/MyDrive/Colab Notebooks/IG_Forecaster/..gitignore
